In [ ]:
!pip install fastapi uvicorn python-multipart pyngrok nest-asyncio mediapipe opencv-python tensorflow==2.12.0 mtcnn

INFO: pip is looking at multiple versions of jax to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of jax to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 586.0/586.0 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 MB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━

In [ ]:
import numpy as np
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.responses import StreamingResponse
import cv2
import io
import nest_asyncio
from pyngrok import ngrok
import uvicorn
import mediapipe as mp

# Initialize FastAPI app
app = FastAPI()

# Initialize Mediapipe Face Detection and Face Mesh with improved settings for side views
mp_face_detection = mp.solutions.face_detection
mp_face_mesh = mp.solutions.face_mesh
mp_drawing = mp.solutions.drawing_utils

# Configure face detection with higher confidence threshold for reliability
face_detection = mp_face_detection.FaceDetection(
    min_detection_confidence=0.5,
    model_selection=1  # model_selection=1 for full range head detection
)

# Configure face mesh with multiple face support and side view detection
face_mesh = mp_face_mesh.FaceMesh(
    static_image_mode=True,
    max_num_faces=5,  # Support multiple faces
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# Define eye region landmarks - focused on eyes, eyebrows and orbital bones
EYE_REGION_INDICES = [
    # Left eye
    33, 7, 163, 144, 145, 153, 154, 155, 133, 173, 157, 158, 159, 160, 161, 246,
    # Right eye
    263, 249, 390, 373, 374, 380, 381, 382, 362, 398, 384, 385, 386, 387, 388, 466,
    # Left eyebrow
    70, 63, 105, 66, 107, 55, 65, 52, 53, 46,
    # Right eyebrow
    336, 296, 334, 293, 300, 285, 295, 282, 283, 276,
    # Orbital bones (eye sockets)
    276, 282, 283, 285, 293, 295, 296, 300, 334, 336, 46, 52, 53, 55, 63, 65, 66, 70, 105, 107
]

# Lower eye landmarks to ensure full eye coverage
LOWER_EYE_INDICES = [
    # Lower left eye
    145, 153, 154, 155, 133, 173, 157, 158, 159, 160, 161, 246,
    # Lower right eye
    374, 380, 381, 382, 362, 398, 384, 385, 386, 387, 388, 466,
    # Lower orbital bones
    111, 117, 118, 119, 120, 121, 128, 245,  # Left lower orbital
    346, 347, 348, 349, 350, 357, 465        # Right lower orbital
]

# Define ear landmark indices for MediaPipe Face Mesh
LEFT_EAR_INDICES = [
    356, 454, 323, 361, 288, 397, 365, 379, 378, 400, 377, 152, 148, 176, 149, 150, 136, 172, 58, 132, 93, 234,
    127, 162, 21, 54, 103, 67, 109, 10
]

RIGHT_EAR_INDICES = [
    127, 234, 93, 132, 58, 172, 136, 150, 149, 176, 148, 152, 377, 400, 378, 379, 365, 397, 288, 361, 323, 454,
    356, 389, 251, 284, 332, 297, 338, 332
]

# Add temporal bone landmarks (area between ear and eye)
LEFT_TEMPORAL_INDICES = [447, 323, 330, 347, 348, 349, 330, 454, 356, 264, 372, 383, 300, 293, 334, 296, 336]
RIGHT_TEMPORAL_INDICES = [227, 137, 177, 215, 138, 135, 227, 127, 234, 93, 132, 58, 172, 136, 150, 149, 176]

# Landmarks for the sides of the face (temple to ear region)
LEFT_SIDE_FACE_INDICES = [
    356, 454, 323, 361, 288, 397, 365, 379, 378, 400, 377, 152, 148, 176, 149, 150, 136, 172, 58, 132, 93, 234,
    127, 162, 21, 54, 103, 67, 109, 10, 338, 297, 332, 284, 251, 389
]

RIGHT_SIDE_FACE_INDICES = [
    127, 234, 93, 132, 58, 172, 136, 150, 149, 176, 148, 152, 377, 400, 378, 379, 365, 397, 288, 361, 323, 454,
    356, 389, 251, 284, 332, 297, 338, 332, 10, 109, 67, 103, 54, 21
]

def detect_faces_mediapipe(image):
    """
    Detect faces using MediaPipe Face Detection
    """
    try:
        # Convert image to RGB (OpenCV uses BGR by default)
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Detect faces
        results = face_detection.process(image_rgb)

        # Extract bounding boxes
        faces = []
        if results.detections:
            for detection in results.detections:
                bboxC = detection.location_data.relative_bounding_box
                ih, iw, _ = image.shape
                x, y = int(bboxC.xmin * iw), int(bboxC.ymin * ih)
                w, h = int(bboxC.width * iw), int(bboxC.height * ih)

                # Ensure bounding box is within image dimensions
                x, y = max(0, x), max(0, y)
                w, h = min(w, iw - x), min(h, ih - y)

                faces.append((x, y, w, h))

        return faces, image_rgb
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Error detecting faces: {str(e)}")

def is_side_view(face_landmarks, image_width):
    """
    Detect if face is in side view and which side the face is looking towards
    Returns: (face_looking_left, face_looking_right)
    """
    try:
        # Get all x coordinates
        all_x = [landmark.x for landmark in face_landmarks.landmark]

        # Calculate the distribution of landmarks across the face width
        left_half_count = sum(1 for x in all_x if x < 0.5)
        right_half_count = sum(1 for x in all_x if x >= 0.5)

        # If landmarks are heavily skewed to one side, it's likely a side view
        total_landmarks = len(all_x)
        threshold = 0.65  # 65% of landmarks on one side indicates side view

        # When more landmarks are on the left side of the image,
        # the face is looking to the right (and vice versa)
        face_looking_right = left_half_count / total_landmarks > threshold
        face_looking_left = right_half_count / total_landmarks > threshold

        # Additional check for nose orientation (more reliable for side view detection)
        # Nose tip is landmark 4
        if 4 < len(face_landmarks.landmark):
            nose_tip_x = face_landmarks.landmark[4].x

            # If nose is on the left side of the image, face is looking right
            if nose_tip_x < 0.35:  # Nose is on the left side
                face_looking_right = True
                face_looking_left = False
            # If nose is on the right side of the image, face is looking left
            elif nose_tip_x > 0.65:  # Nose is on the right side
                face_looking_left = True
                face_looking_right = False

        return face_looking_left, face_looking_right
    except:
        # Default to checking both sides if the specific check fails
        return False, False

def get_eye_region_with_ears(image, face_landmarks, face_looking_left=False, face_looking_right=False):
    """
    Get eye region including orbital bones and stretch horizontally to ears
    """
    try:
        h, w = image.shape[:2]

        # Extract eye region landmarks
        eye_points = []
        for idx in EYE_REGION_INDICES:
            if idx < len(face_landmarks.landmark):
                lm = face_landmarks.landmark[idx]
                # Only use visible landmarks
                if not hasattr(lm, 'visibility') or lm.visibility > 0.5:
                    x, y = int(lm.x * w), int(lm.y * h)
                    eye_points.append((x, y))

        # Extract lower eye landmarks to ensure full coverage
        lower_eye_points = []
        for idx in LOWER_EYE_INDICES:
            if idx < len(face_landmarks.landmark):
                lm = face_landmarks.landmark[idx]
                if not hasattr(lm, 'visibility') or lm.visibility > 0.5:
                    x, y = int(lm.x * w), int(lm.y * h)
                    lower_eye_points.append((x, y))

        # Combine all eye points
        all_eye_points = eye_points + lower_eye_points

        # If we have enough landmarks, calculate the region
        if len(all_eye_points) >= 4:
            x_coords = [p[0] for p in all_eye_points]
            y_coords = [p[1] for p in all_eye_points]

            # Get the bounding box
            eye_x_min = max(0, min(x_coords))
            eye_x_max = min(w, max(x_coords))
            eye_y_min = max(0, min(y_coords))
            eye_y_max = min(h, max(y_coords))

            # Calculate region dimensions
            eye_width = eye_x_max - eye_x_min
            eye_height = eye_y_max - eye_y_min

            # Add padding - more above (for eyebrows) and below (for orbital rim)
            eye_y_min = max(0, eye_y_min - int(eye_height * 0.3))  # 30% padding above
            eye_y_max = min(h, eye_y_max + int(eye_height * 0.5))  # 50% padding below to ensure full eye coverage

            # For front view, stretch horizontally to ears
            if not face_looking_left and not face_looking_right:
                # Get all face landmarks to find face width
                all_x = [int(landmark.x * w) for landmark in face_landmarks.landmark]
                face_left = max(0, min(all_x))
                face_right = min(w, max(all_x))

                # Stretch to full face width with minimal margin
                eye_x_min = face_left + int((face_right - face_left) * 0.02)  # 2% from left edge
                eye_x_max = face_right - int((face_right - face_left) * 0.02)  # 2% from right edge
            else:
                # For side views, add more horizontal padding
                eye_x_min = max(0, eye_x_min - int(eye_width * 0.3))  # Increased from 0.2 to 0.3
                eye_x_max = min(w, eye_x_max + int(eye_width * 0.3))  # Increased from 0.2 to 0.3

            return (eye_x_min, eye_y_min, eye_x_max, eye_y_max)

        # Fallback: use a more conservative approach for the upper face
        all_x = [int(landmark.x * w) for landmark in face_landmarks.landmark]
        all_y = [int(landmark.y * h) for landmark in face_landmarks.landmark]

        face_left = max(0, min(all_x))
        face_right = min(w, max(all_x))
        face_top = max(0, min(all_y))
        face_bottom = min(h, max(all_y))

        face_height = face_bottom - face_top
        face_width = face_right - face_left

        # Define eye region as upper 40% of face
        eye_y_min = face_top
        eye_y_max = face_top + int(face_height * 0.4)  # Cover more of the eye area

        # For front view, stretch horizontally to almost full face width
        if not face_looking_left and not face_looking_right:
            eye_x_min = face_left + int(face_width * 0.02)  # 2% from left edge
            eye_x_max = face_right - int(face_width * 0.02)  # 2% from right edge
        else:
            # For side views, use a more focused region but still wider
            eye_x_min = face_left + int(face_width * 0.05)  # Reduced from 0.1 to 0.05
            eye_x_max = face_right - int(face_width * 0.05)  # Reduced from 0.1 to 0.05

        return (eye_x_min, eye_y_min, eye_x_max, eye_y_max)
    except Exception as e:
        # Ultimate fallback: return None if we can't determine eye region
        return None

def get_ear_region(face_landmarks, image_shape, face_looking_left, face_looking_right):
    """
    Get ear region for side views with improved coverage

    IMPORTANT CLARIFICATION:
    - When face_looking_right=True: The person is looking to THEIR right
      The LEFT side of their face is visible to us, so their LEFT ear is visible
      This LEFT ear appears on the RIGHT side of the image

    - When face_looking_left=True: The person is looking to THEIR left
      The RIGHT side of their face is visible to us, so their RIGHT ear is visible
      This RIGHT ear appears on the LEFT side of the image
    """
    try:
        h, w = image_shape[:2]
        ear_landmarks = []

        # When face is looking to THEIR left, THEIR RIGHT ear is visible to us
        # This RIGHT ear appears on the LEFT side of the image
        if face_looking_left:
            # Use RIGHT ear landmarks (the ear that's visible when face looks left)
            for idx in RIGHT_EAR_INDICES + RIGHT_SIDE_FACE_INDICES:
                if idx < len(face_landmarks.landmark):
                    lm = face_landmarks.landmark[idx]
                    # Only use visible landmarks
                    if not hasattr(lm, 'visibility') or lm.visibility > 0.5:
                        x, y = int(lm.x * w), int(lm.y * h)
                        ear_landmarks.append((x, y))

        # When face is looking to THEIR right, THEIR LEFT ear is visible to us
        # This LEFT ear appears on the RIGHT side of the image
        if face_looking_right:
            # Use LEFT ear landmarks (the ear that's visible when face looks right)
            for idx in LEFT_EAR_INDICES + LEFT_SIDE_FACE_INDICES:
                if idx < len(face_landmarks.landmark):
                    lm = face_landmarks.landmark[idx]
                    # Only use visible landmarks
                    if not hasattr(lm, 'visibility') or lm.visibility > 0.5:
                        x, y = int(lm.x * w), int(lm.y * h)
                        ear_landmarks.append((x, y))

        # If we have enough ear landmarks, calculate the region
        if len(ear_landmarks) >= 3:
            x_coords = [p[0] for p in ear_landmarks]
            y_coords = [p[1] for p in ear_landmarks]

            # Get the bounding box
            ear_x_min = max(0, min(x_coords))
            ear_x_max = min(w, max(x_coords))
            ear_y_min = max(0, min(y_coords))
            ear_y_max = min(h, max(y_coords))

            # Add moderate padding based on ear size
            ear_width = ear_x_max - ear_x_min
            ear_height = ear_y_max - ear_y_min

            # Add more padding for side views to ensure full ear coverage
            ear_x_min = max(0, ear_x_min - int(ear_width * 0.25))
            ear_x_max = min(w, ear_x_max + int(ear_width * 0.25))
            ear_y_min = max(0, ear_y_min - int(ear_height * 0.25))
            ear_y_max = min(h, ear_y_max + int(ear_height * 0.25))

            return (ear_x_min, ear_y_min, ear_x_max, ear_y_max)

        # Fallback: estimate ear position from face bounds
        if face_looking_left or face_looking_right:
            # Get face bounds
            all_x = [int(landmark.x * w) for landmark in face_landmarks.landmark]
            all_y = [int(landmark.y * h) for landmark in face_landmarks.landmark]

            face_left = max(0, min(all_x))
            face_right = min(w, max(all_x))
            face_top = max(0, min(all_y))
            face_bottom = min(h, max(all_y))

            face_width = face_right - face_left
            face_height = face_bottom - face_top

            # Estimate ear region based on face orientation
            if face_looking_left:  # Person looking to THEIR left, RIGHT ear visible on LEFT of image
                # Place ear region on LEFT side of image
                ear_x_min = max(0, face_left - int(face_width * 0.2))
                ear_x_max = face_left + int(face_width * 0.3)
            else:  # face_looking_right - Person looking to THEIR right, LEFT ear visible on RIGHT of image
                # Place ear region on RIGHT side of image
                ear_x_min = face_right - int(face_width * 0.3)
                ear_x_max = min(w, face_right + int(face_width * 0.2))

            # Ensure ear region covers from mid-face height to lower face
            ear_y_min = face_top + int(face_height * 0.25)  # Start higher
            ear_y_max = face_top + int(face_height * 0.8)   # End lower

            return (ear_x_min, ear_y_min, ear_x_max, ear_y_max)

        return None
    except:
        return None



def anonymize_region(image, region, method="pixelate"):
    """
    Anonymize a region using either blur, pixelation, or solid color
    """
    if image is None or region is None:
        return image

    # Create a copy of the image to avoid modifying the original
    anonymized_image = image.copy()

    x_min, y_min, x_max, y_max = region

    # Ensure region is valid
    if x_min >= x_max or y_min >= y_max:
        return anonymized_image

    # Extract the region
    area = anonymized_image[y_min:y_max, x_min:x_max]

    if area.size == 0:
        return anonymized_image

    # Get region dimensions
    region_height, region_width = area.shape[:2]

    # Adjust anonymization strength based on region size
    # This helps with both small and large images
    region_size = region_width * region_height
    image_size = image.shape[0] * image.shape[1]
    size_ratio = region_size / image_size

    # Stronger anonymization for smaller regions (relative to image)
    strength_factor = max(1.0, 0.05 / max(size_ratio, 0.001))

    if method == "blur":
        # Calculate kernel size based on region size and strength factor
        # For small regions in large images, use proportionally larger kernel
        # For large regions, use a more moderate kernel
        base_kernel_size = max(5, min(region_width, region_height) // 10)
        kernel_size = max(5, min(51, int(base_kernel_size * min(3.0, strength_factor))))

        # Ensure kernel size is odd
        kernel_size = kernel_size + 1 if kernel_size % 2 == 0 else kernel_size

        # Apply blur
        area = cv2.GaussianBlur(area, (kernel_size, kernel_size), 0)

    elif method == "pixelate":
        # Calculate pixel size based on region size and strength factor
        base_pixel_size = max(3, min(region_width, region_height) // 15)
        pixel_size = max(3, min(30, int(base_pixel_size * min(2.0, strength_factor))))

        # Ensure we don't divide by zero
        if pixel_size > 0 and region_width > pixel_size and region_height > pixel_size:
            # Resize down to create pixelation effect
            temp = cv2.resize(area,
                            (max(1, region_width // pixel_size),
                            max(1, region_height // pixel_size)),
                            interpolation=cv2.INTER_LINEAR)
            # Resize back up to original size with nearest neighbor interpolation
            area = cv2.resize(temp, (region_width, region_height), interpolation=cv2.INTER_NEAREST)

    elif method == "solid":
        # Create a solid black bar
        area = np.zeros_like(area)

    # Replace the original region with the anonymized version
    anonymized_image[y_min:y_max, x_min:x_max] = area

    return anonymized_image

@app.post("/anonymize/")
async def anonymize_image(file: UploadFile = File(...), method: str = "solid"):
    try:
        # Validate method parameter
        if method not in ["blur", "pixelate", "solid"]:
            method = "pixelate"  # Default to pixelate if invalid method

        # Read the uploaded image file
        contents = await file.read()
        image = cv2.imdecode(np.frombuffer(contents, np.uint8), cv2.IMREAD_COLOR)

        if image is None or image.size == 0:
            raise HTTPException(status_code=400, detail="Invalid image file")

        # Check if image is very large and resize if necessary for processing
        # but keep track of original size for final output
        max_dimension = 2000  # Maximum dimension for processing
        h, w = image.shape[:2]

        # Store original size for potential upscaling later
        original_size = (w, h)
        resized = False

        if max(h, w) > max_dimension:
            # Calculate new dimensions while preserving aspect ratio
            if w > h:
                new_w = max_dimension
                new_h = int(h * (max_dimension / w))
            else:
                new_h = max_dimension
                new_w = int(w * (max_dimension / h))

            # Resize image for processing
            image = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_AREA)
            resized = True

        # For very small images, temporarily upscale for better landmark detection
        min_dimension = 200  # Minimum dimension for reliable processing
        temp_upscaled = False
        temp_original_size = (w, h)

        if max(h, w) < min_dimension:
            # Calculate new dimensions while preserving aspect ratio
            scale_factor = min_dimension / max(h, w)
            new_w = int(w * scale_factor)
            new_h = int(h * scale_factor)

            # Temporarily upscale image for processing
            image = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
            temp_upscaled = True

        # Convert to RGB for processing
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Process with face mesh
        results = face_mesh.process(image_rgb)
        anonymized_image = image_rgb.copy()

        faces_processed = False

        if results.multi_face_landmarks:
            for face_landmarks in results.multi_face_landmarks:
                # Determine if face is in side view
                # Returns (face_looking_left, face_looking_right)
                face_looking_left, face_looking_right = is_side_view(face_landmarks, image_rgb.shape[1])

                # Get eye region including orbital bones and stretched to ears for front view
                eye_region = get_eye_region_with_ears(image_rgb, face_landmarks, face_looking_left, face_looking_right)

                # Only anonymize if we have a valid eye region
                if eye_region:
                    anonymized_image = anonymize_region(anonymized_image, eye_region, method=method)
                    faces_processed = True

                # For side views, also anonymize the ear region
                if face_looking_left or face_looking_right:
                    ear_region = get_ear_region(face_landmarks, image_rgb.shape, face_looking_left, face_looking_right)
                    if ear_region:
                        anonymized_image = anonymize_region(anonymized_image, ear_region, method=method)
                        faces_processed = True

        # If no faces were processed with face mesh, fall back to simple face detection
        if not faces_processed:
            # Fallback to simple face detection
            faces, _ = detect_faces_mediapipe(image)
            if faces:
                for (x, y, w, h) in faces:
                    # For small faces, be more precise about the eye region
                    # to avoid covering the whole face
                    face_size_ratio = (w * h) / (image.shape[0] * image.shape[1])

                    if face_size_ratio < 0.1:  # Small face relative to image
                        # More precise eye region - just the upper part
                        eye_y_min = y + int(h * 0.1)  # Start a bit below the top
                        eye_y_max = y + int(h * 0.4)  # Cover eyes and brows plus more below

                        # For small faces, still stretch horizontally to approximate ear width
                        eye_x_min = max(0, x - int(w * 0.1))   # Extend beyond face left
                        eye_x_max = min(image.shape[1], x + w + int(w * 0.1))  # Extend beyond face right
                    else:
                        # Standard eye region for normal-sized faces - stretch horizontally to ears
                        eye_y_min = y
                        eye_y_max = y + int(h * 0.4)  # Upper 40% of face (increased from 35%)

                        # Stretch horizontally to include ears
                        eye_x_min = max(0, x - int(w * 0.1))
                        eye_x_max = min(image.shape[1], x + w + int(w * 0.1))

                    # Anonymize eye region
                    eye_region = (eye_x_min, eye_y_min, eye_x_max, eye_y_max)
                    anonymized_image = anonymize_region(anonymized_image, eye_region, method=method)


                    # Side view ear detection logic with explicit comments
                    # Check if face is near left or right edge of image
                    is_near_left_edge = x < image.shape[1] * 0.2
                    is_near_right_edge = (x + w) > image.shape[1] * 0.8

                    # If face is near left edge of image:
                    # - The face is likely looking toward the right
                    # - The person's right ear is visible
                    # - This ear appears on the LEFT side of the image
                    if is_near_left_edge:
                        # Define ear region on LEFT side of image
                        ear_x_min = max(0, x - int(w * 0.3))  # Start further left of face
                        ear_x_max = x + int(w * 0.2)          # End slightly into face
                        ear_y_min = y + int(h * 0.25)         # Start at upper quarter of face
                        ear_y_max = y + int(h * 0.8)          # End at lower part of face

                        ear_region = (ear_x_min, ear_y_min, ear_x_max, ear_y_max)
                        anonymized_image = anonymize_region(anonymized_image, ear_region, method=method)

                    # If face is near right edge of image:
                    # - The face is likely looking toward the left
                    # - The person's left ear is visible
                    # - This ear appears on the RIGHT side of the image
                    if is_near_right_edge:
                        # Define ear region on RIGHT side of image
                        ear_x_min = x + w - int(w * 0.2)                  # Start slightly inside face
                        ear_x_max = min(image.shape[1], x + w + int(w * 0.3))  # End further right of face
                        ear_y_min = y + int(h * 0.25)                     # Start at upper quarter of face
                        ear_y_max = y + int(h * 0.8)                      # End at lower part of face

                        ear_region = (ear_x_min, ear_y_min, ear_x_max, ear_y_max)
                        anonymized_image = anonymize_region(anonymized_image, ear_region, method=method)
            else:
                raise HTTPException(status_code=400, detail="No faces detected")

        # If image was temporarily upscaled for processing, revert to original size
        if temp_upscaled:
            anonymized_image = cv2.resize(anonymized_image, temp_original_size, interpolation=cv2.INTER_LINEAR)

        # If image was resized, scale it back to original size
        if resized:
            anonymized_image = cv2.resize(anonymized_image, original_size, interpolation=cv2.INTER_LINEAR)

        # Convert the anonymized image to bytes
        _, img_encoded = cv2.imencode(".jpg", cv2.cvtColor(anonymized_image, cv2.COLOR_RGB2BGR))
        return StreamingResponse(io.BytesIO(img_encoded.tobytes()), media_type="image/jpeg")

    except HTTPException as he:
        raise he
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Error processing image: {str(e)}")


# Allow nested asyncio loops (required for running FastAPI in Colab)
nest_asyncio.apply()

# Authenticate Ngrok with your authtoken
ngrok.set_auth_token("2sozVqDRkdN8NKnwGBSyfRXUQ7c_5XJpniSgSz6SnNbLLcmhS")  # Replace with your actual Ngrok authtoken

# Start the FastAPI server
ngrok_tunnel = ngrok.connect(8000)
print("Public URL:", ngrok_tunnel.public_url)

try:
    # Run the FastAPI app
    uvicorn.run(app, host="127.0.0.1", port=8000)
finally:
    # Close the Ngrok tunnel when done
    ngrok.kill()




INFO:     Started server process [827]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


Public URL: https://f18b-34-80-30-84.ngrok-free.app
INFO:     35.221.39.255:0 - "POST /anonymize/ HTTP/1.1" 200 OK
INFO:     35.221.39.255:0 - "POST /anonymize/ HTTP/1.1" 200 OK
INFO:     35.221.39.255:0 - "POST /anonymize/ HTTP/1.1" 200 OK
INFO:     35.221.39.255:0 - "POST /anonymize/ HTTP/1.1" 200 OK
INFO:     35.221.39.255:0 - "POST /anonymize/ HTTP/1.1" 200 OK
INFO:     35.221.39.255:0 - "POST /anonymize/ HTTP/1.1" 400 Bad Request
INFO:     35.221.39.255:0 - "POST /anonymize/ HTTP/1.1" 200 OK
